# convert train to be used with Flagembeddings to generate hard negatives
https://github.com/FlagOpen/FlagEmbedding/tree/master/examples/finetune#hard-negatives

In [2]:
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
 os.getcwd(),
 os.pardir+'/playground')
)
#only add it once
if (PROJECT_ROOT not in sys.path):
 sys.path.append(PROJECT_ROOT)

import pandas as pd
import utils

from datasets import load_dataset
train_dataset = load_dataset("json", data_files="trn.json", split="train")
tst_dataset = load_dataset("json", data_files="tst.json", split="train")

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /home/kperkins411/.cache/huggingface/token
Login successful


In [3]:
import datasets

#rename columns
train_dataset = train_dataset.rename_column("anchor", "query")
train_dataset = train_dataset.rename_column("positive", "pos")

#remove unneeded columns
train_dataset=train_dataset.remove_columns(["id",'most_dissimilar_context'])

#add a new column that consists of empty lists
new_column = [ [] for _ in range(len(train_dataset)) ]
train_dataset=train_dataset.add_column('neg',new_column)

In [5]:
#convert "pos" column to list
train_dataset=utils.change_col_to_list(train_dataset,'pos')

In [6]:
#save it
train_dataset.to_json(f'train_FLAG.jsonl',orient='records',lines=True)

Creating json from Arrow format: 100%|██████████| 36/36 [00:00<00:00, 176.34ba/s]


24098360

In [7]:
print(train_dataset.features)
print(train_dataset)
train_dataset[1]['pos']

{'query': Value(dtype='string', id=None), 'neg': Sequence(feature=Value(dtype='null', id=None), length=-1, id=None), 'pos': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)}
Dataset({
    features: ['query', 'neg', 'pos'],
    num_rows: 35272
})


['Each of the Suppliers warrants that the Products shall comply with the specifications and documentation agreed by the relevant Supplier and the Company in writing that is applicable to such Products for the Warranty Period.']

In [11]:
#select just 10 rows
tds=train_dataset.select(range(10))

In [12]:
type(tds['query'])

list

In [15]:
tds[0]['neg']

[]

In [114]:
tds[0]['pos']

'Information We Collect From Other Sources We may also receive information from other sources and combine that with information we collect through our Services. For example: If you choose to link, create, or log in to your Uber account with a payment provider (e.g., Google Wallet) or social media service (e.g., Facebook), or if you engage with a separate app or website that uses our API (or whose API we use), we may receive information about you or your connections from that site or app.'

In [20]:
#show first 2 rows
tds[0:2]

{'query': ['What safeguards are in place to protect the information obtained from third-party sources?',
  'Is there a guarantee from the manufacturers regarding the conformity of the items to the mutually approved written standards for a certain duration?'],
 'neg': [[], []],
 'pos': [['Information We Collect From Other Sources We may also receive information from other sources and combine that with information we collect through our Services. For example: If you choose to link, create, or log in to your Uber account with a payment provider (e.g., Google Wallet) or social media service (e.g., Facebook), or if you engage with a separate app or website that uses our API (or whose API we use), we may receive information about you or your connections from that site or app.'],
  ['Each of the Suppliers warrants that the Products shall comply with the specifications and documentation agreed by the relevant Supplier and the Company in writing that is applicable to such Products for the Warra

In [21]:
tds.features

{'query': Value(dtype='string', id=None),
 'neg': Sequence(feature=Value(dtype='null', id=None), length=-1, id=None),
 'pos': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)}